# Pipeline de Co-retweet — execução ponta a ponta

Roda os seis módulos da pipeline sobre **todos os CSVs de um evento** e gera o grafo
de comunidades. Para trocar o evento, altere `EVENTO` na célula de configuração.

Etapas: **M1** carga → **M2** filtragem → **M3** matriz bipartida → **M4** projeção Jaccard
→ **M5** backbone (τ) → **M6** comunidades (Leiden). Cada etapa persiste seus artefatos em
`data/processed/<evento>/` e tem uma célula de inspeção preliminar.

**Cache por estágio (idempotência).** Ao re-rodar, cada módulo (`.run(...)`) carrega do
disco se os artefatos já existem, em vez de recalcular. Para forçar o recálculo a partir de
um módulo (e todos os posteriores), ajuste `FORCE_FROM` na célula de configuração.

In [1]:
import sys
from pathlib import Path

# Raiz do projeto: o notebook vive em notebooks/, mas pode rodar de notebooks/
# (Jupyter) ou da raiz do repositório (nbconvert/CI).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))  # habilita `from modules.* import ...`

import json
import numpy as np
import pandas as pd
import scipy.sparse as sp

from modules.load import RetweetLoader
from modules.filter import NoiseFilter
from modules.bipartite import BipartiteBuilder
from modules.project import JaccardProjector
from modules.backbone import BackboneExtractor
from modules.community import CommunityDetector

## Configuração

Troque `EVENTO` para rodar a pipeline em outro conjunto de dados. Os parâmetros
em aberto (`N`, `τ`) devem ser calibrados em `notebooks/exploratory-analysis.ipynb`
antes de rodar em escala.

In [2]:
# ---- TROQUE AQUI O EVENTO ----
# Opções (subpastas de data/raw/): "invasao-3-poderes", "eleicoes",
#                                   "roberto-jefferson", "mobilizacao-0709"
EVENTO = "invasao-3-poderes"

# ---- Parâmetros da pipeline (calibrar antes de rodar em escala) ----
MIN_USER_RETWEETS = 10       # N  — filtro de atividade (mín. de retweets por usuário)
TAU = 0.10                   # τ  — peso Jaccard mínimo no backbone
RESOLUTION = 1.0             # resolução do Leiden

# ---- Cache por estágio ----
# Cada módulo só recalcula se faltar artefato OU se for forçado aqui.
# FORCE_FROM = None  -> usa cache onde houver (cold start calcula sozinho).
# FORCE_FROM = N     -> recalcula o módulo N e TODOS os posteriores (cascata).
# Mudou o código de um módulo? Aponte FORCE_FROM para ele.
FORCE_FROM = 2

def _force(n: int) -> bool:
    return FORCE_FROM is not None and n >= FORCE_FROM

RAW_DIR = PROJECT_ROOT / "data" / "raw" / EVENTO
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / EVENTO
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(), f"Pasta do evento não encontrada: {RAW_DIR}"
csvs = sorted(RAW_DIR.glob("*.csv"))
print(f"Evento: {EVENTO}")
print(f"Raw:       {RAW_DIR}  ({len(csvs)} CSVs)")
for c in csvs:
    print(f"  - {c.name}")
print(f"Processed: {PROCESSED_DIR}")
print(f"Parâmetros: N={MIN_USER_RETWEETS}, τ={TAU}, resolution={RESOLUTION}")
print(f"FORCE_FROM: {FORCE_FROM}")

Evento: invasao-3-poderes
Raw:       /home/vinicius/tcc/data/raw/invasao-3-poderes  (7 CSVs)
  - 0801_invasao-06hr-09hr.csv
  - 0801_invasao-09hr-12hr.csv
  - 0801_invasao-12hr-15hr.csv
  - 0801_invasao-15hr-18hr.csv
  - 0801_invasao-18hr-21hr.csv
  - 0801_invasao-21hr-01hr.csv
  - 0901_invasao-01hr-06hr.csv
Processed: /home/vinicius/tcc/data/processed/invasao-3-poderes
Parâmetros: N=10, τ=0.1, resolution=1.0
FORCE_FROM: 2


## Módulo 1 — Carga de retweets

Lê todos os CSVs do evento e mantém apenas os retweets.

In [3]:
retweets = RetweetLoader(RAW_DIR).run(out_dir=PROCESSED_DIR, force=_force(1))
print(f"Retweets carregados: {len(retweets):,}")
retweets.head()

[cache] RetweetLoader: hit
Retweets carregados: 1,100,026


,author_id,referenced_tweet_id,created_at
0,1146896004341489675,1611790880922341376,2023-01-08 09:00:02
1,713194373412954112,1611938142445076483,2023-01-08 09:00:03
2,1034053475867533312,1611922426308296704,2023-01-08 09:00:04
3,1567983664423735301,1612009274473074688,2023-01-08 09:00:05
4,414776107,1611901031058743301,2023-01-08 09:00:13


In [4]:
# Inspeção M1
print(f"Linhas (retweets):  {len(retweets):,}")
print(f"Usuários únicos:    {retweets['author_id'].nunique():,}")
print(f"Tweets únicos:      {retweets['referenced_tweet_id'].nunique():,}")
print(f"Período:            {retweets['created_at'].min()}  →  {retweets['created_at'].max()}")

Linhas (retweets):  1,100,026
Usuários únicos:    321,268
Tweets únicos:      29,206
Período:            2023-01-08 09:00:02  →  2023-01-09 08:59:59


## Módulo 2 — Filtragem de ruído

Remove usuários inativos (< N retweets). Tweets virais são preservados (ver D5).

In [5]:
nf = NoiseFilter(min_user_retweets=MIN_USER_RETWEETS)
filtered = nf.run(retweets, out_dir=PROCESSED_DIR, force=_force(2))
print(f"Retweets após filtro: {len(filtered):,}")
filtered.head()

[cache] NoiseFilter: forçado
Retweets após filtro: 520,582


,author_id,referenced_tweet_id,created_at
0,713194373412954112,1611938142445076483,2023-01-08 09:00:03
1,1034053475867533312,1611922426308296704,2023-01-08 09:00:04
2,26663913,1611863586766299137,2023-01-08 09:00:21
3,1041864641486442497,1611904042145402887,2023-01-08 09:00:22
4,1592230020923138050,1611979214927822848,2023-01-08 09:00:26


In [6]:
# Inspeção M2
print(json.dumps(nf.stats, indent=2, ensure_ascii=False))
ret = len(filtered) / len(retweets) if len(retweets) else 0
print(f"\nRetenção de linhas: {len(retweets):,} -> {len(filtered):,} ({ret:.1%})")

{
  "initial": {
    "rows": 1100026,
    "users": 321268,
    "tweets": 29206
  },
  "after_user_filter": {
    "rows": 520582,
    "users": 22217,
    "tweets": 19386
  }
}

Retenção de linhas: 1,100,026 -> 520,582 (47.3%)


## Módulo 3 — Matriz bipartida usuário × tweet

In [7]:
bg = BipartiteBuilder().run(filtered, out_dir=PROCESSED_DIR, force=_force(3))
print("Matriz B:", bg.B.shape, "| nnz:", f"{bg.B.nnz:,}")

[cache] BipartiteBuilder: forçado
Matriz B: (22217, 19386) | nnz: 520,574


In [8]:
# Inspeção M3
n_u, n_t = bg.B.shape
print(f"Usuários (linhas): {n_u:,}")
print(f"Tweets (colunas):  {n_t:,}")
print(f"Não-zeros:         {bg.B.nnz:,}")
print(f"Densidade:         {bg.B.nnz / (n_u * n_t):.2e}")
col_sum = np.asarray(bg.B.sum(axis=0)).ravel()   # retweetadores por tweet sobrevivente
print(f"Máx retweetadores num tweet sobrevivente: {int(col_sum.max()):,}")

Usuários (linhas): 22,217
Tweets (colunas):  19,386
Não-zeros:         520,574
Densidade:         1.21e-03
Máx retweetadores num tweet sobrevivente: 3,490


## Pré-voo da projeção (custo de memória)

⚠️ O custo da projeção `B·Bᵀ` cresce com o **quadrado** do nº de retweetadores do tweet
mais denso. Como os tweets virais são preservados (D5), o número de pares co-retweet pode
ser grande demais para materializar a matriz densa de uma vez — por isso a projeção é feita
em blocos, com corte do peso Jaccard durante a geração. Rode a célula abaixo **antes** do
Módulo 4 para dimensionar o custo.

In [9]:
# Pré-voo: limite superior de pares co-retweet (proxy do custo da projeção)
from math import comb

pair_upper = int(sum(comb(int(k), 2) for k in col_sum if k >= 2))
print(f"Limite superior de pares co-retweet: {pair_upper:,}")
if pair_upper > 50_000_000:
    print("\n⚠️  ALTO: materializar B·Bᵀ inteiro estouraria a memória.")
    print("    A projeção em blocos com corte de Jaccard (Módulo 4) mantém o uso")
    print("    de memória limitado e só persiste as arestas acima do limiar.")
else:
    print("OK: custo da projeção dentro do tratável.")

Limite superior de pares co-retweet: 184,394,095

⚠️  ALTO: materializar B·Bᵀ inteiro estouraria a memória.
    A projeção em blocos com corte de Jaccard (Módulo 4) mantém o uso
    de memória limitado e só persiste as arestas acima do limiar.


## Módulo 4 — Projeção Jaccard (grafo de co-retweet)

In [10]:
pg = JaccardProjector().run(bg, out_dir=PROCESSED_DIR, force=_force(4))
print("Grafo projetado:", pg.W.shape, "| arestas:", f"{pg.W.nnz:,}")

[cache] JaccardProjector: forçado
Grafo projetado: (22217, 22217) | arestas: 80,536,314


In [11]:
# Inspeção M4 — distribuição dos pesos Jaccard (ajuda a escolher τ)
w = pg.W.data
print(f"Nós:     {pg.W.shape[0]:,}")
print(f"Arestas: {pg.W.nnz:,}")
print(f"Peso Jaccard — min/méd/máx: {w.min():.3f} / {w.mean():.3f} / {w.max():.3f}")
print("\nArestas retidas por limiar τ:")
for t in (0.05, 0.10, 0.15, 0.20):
    print(f"  τ >= {t:.2f}:  {int((w >= t).sum()):,}  ({(w >= t).mean():.1%})")

Nós:     22,217
Arestas: 80,536,314
Peso Jaccard — min/méd/máx: 0.002 / 0.048 / 1.000

Arestas retidas por limiar τ:
  τ >= 0.05:  28,327,946  (35.2%)
  τ >= 0.10:  5,661,890  (7.0%)
  τ >= 0.15:  951,223  (1.2%)
  τ >= 0.20:  173,497  (0.2%)


## Módulo 5 — Backbone (universal threshold τ)

In [12]:
bb = BackboneExtractor(tau=TAU)
pg_bb = bb.run(pg, out_dir=PROCESSED_DIR, force=_force(5))
print("Backbone:", pg_bb.W.shape, "| arestas:", f"{pg_bb.W.nnz:,}")

[cache] BackboneExtractor: forçado
Backbone: (22097, 22097) | arestas: 5,661,890


In [13]:
# Inspeção M5
print(json.dumps(bb.stats, indent=2))
print(f"\nNós:     {bb.stats['before']['nodes']:,} -> {bb.stats['after']['nodes']:,}")
print(f"Arestas: {bb.stats['before']['edges']:,} -> {bb.stats['after']['edges']:,}")

{
  "before": {
    "nodes": 22217,
    "edges": 80536314
  },
  "after": {
    "nodes": 22097,
    "edges": 5661890
  }
}

Nós:     22,217 -> 22,097
Arestas: 80,536,314 -> 5,661,890


## Módulo 6 — Detecção de comunidades (Leiden)

In [14]:
cr = CommunityDetector(resolution=RESOLUTION).run(pg_bb, out_dir=PROCESSED_DIR, force=_force(6))
print(f"Comunidades: {len(set(cr.membership))}")
print(f"Modularidade: {cr.partition.modularity:.4f}")

[cache] CommunityDetector: forçado
Comunidades: 14
Modularidade: 0.5034


In [15]:
# Inspeção M6
sizes = pd.Series(cr.membership).value_counts()
print(f"Nº de comunidades: {len(sizes)}")
print(f"Modularidade:      {cr.partition.modularity:.4f}")
print(f"Nós no grafo:      {cr.g.vcount():,} | arestas: {cr.g.ecount():,}")
print("\nTamanho das 15 maiores comunidades:")
print(sizes.head(15).to_string())
if sizes.sum():
    print(f"\nTop 5 comunidades cobrem {sizes.head(5).sum() / sizes.sum():.1%} dos nós")

Nº de comunidades: 14
Modularidade:      0.5034
Nós no grafo:      22,097 | arestas: 5,661,890

Tamanho das 15 maiores comunidades:
1     10387
0      7541
2      4128
3         8
4         6
7         6
5         5
9         3
10        3
6         2
8         2
11        2
12        2
13        2

Top 5 comunidades cobrem 99.9% dos nós


## Artefatos salvos

In [16]:
print(f"Arquivos em {PROCESSED_DIR}:\n")
for f in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {f.name:34s} {f.stat().st_size / 1024:>10,.1f} KB")

Arquivos em /home/vinicius/tcc/data/processed/invasao-3-poderes:

  backbone_W.npz                       18,122.7 KB
  backbone_stats.json                       0.1 KB
  backbone_user_index.parquet             335.9 KB
  bipartite_B.npz                         896.0 KB
  bipartite_tweet_index.parquet           277.9 KB
  bipartite_user_index.parquet            337.7 KB
  community_graph.graphml             572,690.2 KB
  filter_stats.json                         0.2 KB
  filtered_retweets.parquet             3,109.1 KB
  hydrated_tweets.csv                      55.1 KB
  hydrated_tweets.jsonl                    69.7 KB
  hydrated_users.json                      69.6 KB
  membership.parquet                      344.6 KB
  original_tweets.csv                   6,114.8 KB
  projection_W.npz                    240,827.4 KB
  projection_user_index.parquet           337.7 KB
  retweets.parquet                     15,777.7 KB
